# Easy Run EDA Notebook

This notebook performs exploratory data analysis for easy-run performance metrics.

Primary goals:
- Understand trends in aerobic efficiency and durability.
- Inspect data quality and missingness.
- Explore metric relationships and outliers.
- Build a lightweight baseline model for score prediction.

## 1. Environment Setup and Imports

Import core data science libraries and configure plotting/display options.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Libraries loaded. scikit-learn available:", SKLEARN_AVAILABLE)

## 2. Notebook Parameters and File Paths

Define configurable parameters, input/output paths, and reusable constants.

In [ ]:
# Paths
REPORT_CSV = Path("../reports/hr_improvement_analysis.csv")
FALLBACK_CSV = Path("../reports/easy/hr_improvement_analysis.csv")
OUTPUT_DIR = Path("../reports/easy")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility and analysis constants
RANDOM_SEED = 42
TEST_SIZE = 0.25
ROLLING_WINDOW = 3
EF_TRAINED_THRESHOLD = 1.8
DECOUPLING_FIT_THRESHOLD = 5.0

KEY_METRICS = [
    "steady_power_per_hr",
    "aerobic_drift_pct",
    "fatigue_resilience_score",
    "steady_pace_min_per_km",
    "easy_run_score",
]

print("Primary CSV:", REPORT_CSV)
print("Fallback CSV:", FALLBACK_CSV)
print("Output dir:", OUTPUT_DIR.resolve())

## 3. Load Dataset

Read the analysis dataset, parse date, and inspect high-level structure.

In [ ]:
data_path = REPORT_CSV if REPORT_CSV.exists() else FALLBACK_CSV

if not data_path.exists():
    raise FileNotFoundError(
        "No HR analysis CSV found. Run: python src/hr_improvement_tracker.py"
    )

df = pd.read_csv(data_path)

if "date" in df.columns:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.sort_values("date")

print("Loaded:", data_path)
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

df.head(5)

## 4. Data Cleaning and Feature Preparation

Check data quality and prepare numeric features for analysis and modeling.

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print("Top missing columns (%):")
print(missing_pct.head(20).round(2))

print("\nDuplicate rows:", int(df.duplicated().sum()))
print("\nDtypes:")
print(df.dtypes.sort_index())

numeric_df = df.select_dtypes(include=[np.number]).copy()
print("\nNumeric summary:")
display(numeric_df.describe().T)

# Minimal preprocessing for model cell
model_df = numeric_df.copy()
if "easy_run_score" in model_df.columns:
    model_df = model_df.dropna(subset=["easy_run_score"])
model_df = model_df.fillna(model_df.median(numeric_only=True))

feature_cols = [c for c in model_df.columns if c != "easy_run_score"]
X = model_df[feature_cols] if feature_cols else pd.DataFrame()
y = model_df["easy_run_score"] if "easy_run_score" in model_df.columns else pd.Series(dtype=float)

print("Model-ready shape:", model_df.shape)
print("Feature count:", len(feature_cols))

## 5. Exploratory Visualizations

Explore distributions, trends, correlations, and zone behavior.

Interpretation guide:
- Lower aerobic_drift_pct is usually better aerobic durability.
- Higher steady_power_per_hr indicates better efficiency at a given HR.
- Lower steady_pace_min_per_km means faster pace.

In [ ]:
available_metrics = [c for c in KEY_METRICS if c in df.columns]
if available_metrics:
    # Histograms + KDE
    n = len(available_metrics)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3.8 * n))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, available_metrics):
        sns.histplot(df[col].dropna(), kde=True, ax=ax, color="steelblue")
        ax.set_title(f"Distribution: {col}")
    plt.tight_layout()
    plt.show()

    # Boxplots
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.5))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, available_metrics):
        sns.boxplot(y=df[col], ax=ax, color="lightcoral")
        ax.set_title(col)
    plt.tight_layout()
    plt.show()
else:
    print("No key metrics found for distribution plots.")

# Time trends
if "date" in df.columns and df["date"].notna().any() and available_metrics:
    trend_cols = [c for c in ["steady_power_per_hr", "aerobic_drift_pct", "fatigue_resilience_score", "steady_pace_min_per_km"] if c in df.columns]
    for col in trend_cols:
        plot_df = df[["date", col]].dropna().sort_values("date")
        if plot_df.empty:
            continue
        plt.figure(figsize=(12, 4))
        plt.plot(plot_df["date"], plot_df[col], marker="o", label=col)
        if len(plot_df) >= ROLLING_WINDOW:
            roll = plot_df[col].rolling(ROLLING_WINDOW).mean()
            plt.plot(plot_df["date"], roll, linestyle="--", linewidth=2, label=f"rolling mean ({ROLLING_WINDOW})")
        if col == "steady_power_per_hr":
            plt.axhline(EF_TRAINED_THRESHOLD, linestyle=":", color="green", label="EF trained threshold")
        if col == "aerobic_drift_pct":
            plt.axhline(DECOUPLING_FIT_THRESHOLD, linestyle=":", color="green", label="Decoupling fit threshold")
        plt.title(f"Trend: {col}")
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("Skipping trend plots (date or key metrics unavailable).")

# Relationship plots
if {"steady_avg_hr", "steady_pace_min_per_km", "steady_power_per_hr"}.issubset(df.columns):
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x="steady_avg_hr", y="steady_pace_min_per_km", hue="steady_power_per_hr", palette="RdYlGn", s=100)
    plt.gca().invert_yaxis()
    plt.title("Steady HR vs Steady Pace (colored by EF)")
    plt.tight_layout()
    plt.show()

if {"aerobic_drift_pct", "fatigue_resilience_score"}.issubset(df.columns):
    plt.figure(figsize=(8, 6))
    sns.regplot(data=df, x="aerobic_drift_pct", y="fatigue_resilience_score", scatter_kws={"s": 80}, line_kws={"color": "crimson"})
    plt.title("Aerobic Drift vs Fatigue Resilience")
    plt.tight_layout()
    plt.show()

if {"steady_power_per_hr", "easy_run_score"}.issubset(df.columns):
    plt.figure(figsize=(8, 6))
    sns.regplot(data=df, x="steady_power_per_hr", y="easy_run_score", scatter_kws={"s": 80}, line_kws={"color": "darkblue"})
    plt.title("EF vs Easy Run Score")
    plt.tight_layout()
    plt.show()

# Correlation heatmap
corr_cols = [c for c in [
    "steady_power_per_hr", "aerobic_drift_pct", "fatigue_resilience_score", "steady_pace_min_per_km",
    "steady_avg_hr", "steady_avg_speed_mps", "steady_hr_rise_bpm", "easy_run_score"
] if c in numeric_df.columns]

if len(corr_cols) >= 2:
    plt.figure(figsize=(10, 7))
    corr = numeric_df[corr_cols].corr()
    sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", square=True)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough columns for correlation heatmap.")

# Zone behavior
hr_zone_cols = [c for c in ["hr_z1_pct", "hr_z2_pct", "hr_z3_pct", "hr_z4_pct", "hr_z5_pct"] if c in df.columns]
power_zone_cols = [c for c in ["power_z1_pct", "power_z2_pct", "power_z3_pct", "power_z4_pct", "power_z5_pct"] if c in df.columns]

if "date" in df.columns and hr_zone_cols:
    z = df[["date"] + hr_zone_cols].dropna().sort_values("date")
    if not z.empty:
        z_plot = z.set_index("date")
        z_plot.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="viridis")
        plt.title("HR Zone Distribution by Run")
        plt.ylabel("% time")
        plt.tight_layout()
        plt.show()

        z_plot.mean().plot(kind="bar", figsize=(8, 4), color="teal")
        plt.title("Average HR Zone Distribution")
        plt.ylabel("Average %")
        plt.tight_layout()
        plt.show()

if "date" in df.columns and power_zone_cols:
    z = df[["date"] + power_zone_cols].dropna().sort_values("date")
    if not z.empty:
        z_plot = z.set_index("date")
        z_plot.plot(kind="bar", stacked=True, figsize=(12, 5), colormap="magma")
        plt.title("Power Zone Distribution by Run")
        plt.ylabel("% time")
        plt.tight_layout()
        plt.show()

        z_plot.mean().plot(kind="bar", figsize=(8, 4), color="purple")
        plt.title("Average Power Zone Distribution")
        plt.ylabel("Average %")
        plt.tight_layout()
        plt.show()

# Weekly aggregation
weekly = pd.DataFrame()
if "date" in df.columns and df["date"].notna().any():
    wdf = df.dropna(subset=["date"]).copy()
    wdf["week"] = wdf["date"].dt.to_period("W").dt.start_time
    weekly_cols = [c for c in ["steady_power_per_hr", "aerobic_drift_pct", "steady_pace_min_per_km"] if c in wdf.columns]
    if weekly_cols:
        weekly = wdf.groupby("week", as_index=False)[weekly_cols].mean()
        display(weekly.tail(10))

        for col in weekly_cols:
            plt.figure(figsize=(11, 4))
            plt.plot(weekly["week"], weekly[col], marker="o")
            plt.title(f"Weekly Mean: {col}")
            plt.tight_layout()
            plt.show()

# Anomaly flags
flag_cols = ["date", "aerobic_drift_pct", "steady_power_per_hr", "fatigue_resilience_score", "easy_run_score"]
flag_cols = [c for c in flag_cols if c in df.columns]

if {"aerobic_drift_pct", "steady_power_per_hr", "fatigue_resilience_score"}.issubset(df.columns):
    ef_p20 = df["steady_power_per_hr"].quantile(0.20)
    flagged = df[
        (df["aerobic_drift_pct"] > 8)
        | (df["steady_power_per_hr"] < ef_p20)
        | (df["fatigue_resilience_score"] < 30)
    ][flag_cols].sort_values("date" if "date" in flag_cols else flag_cols[0])

    print("Flagged runs count:", len(flagged))
    display(flagged)
else:
    print("Skipping anomaly flags (required columns unavailable).")

## 6. Baseline Model Training

Train a simple baseline regressor to estimate easy_run_score from available numeric metrics.

If easy_run_score or scikit-learn is unavailable, this section safely skips.

In [ ]:
model_result = {}

if SKLEARN_AVAILABLE and "easy_run_score" in model_df.columns and len(model_df) >= 8 and len(feature_cols) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )

    model = RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_SEED,
        max_depth=None,
        min_samples_leaf=1,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    r2 = r2_score(y_test, y_pred)

    model_result = {
        "model": model,
        "X_test": X_test,
        "y_test": y_test,
        "y_pred": y_pred,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }

    print("Baseline model trained.")
    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R2:   {r2:.3f}")
else:
    print("Skipping baseline model training. Requirements not met.")
    print("Need scikit-learn, easy_run_score column, non-empty features, and >= 8 rows.")

## 7. Model Evaluation and Artifact Export

Evaluate baseline predictions, inspect residual behavior, and export artifacts for reuse.

In [ ]:
if model_result:
    y_test = model_result["y_test"]
    y_pred = model_result["y_pred"]

    # Predicted vs actual
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, alpha=0.8)
    lo = min(y_test.min(), y_pred.min())
    hi = max(y_test.max(), y_pred.max())
    plt.plot([lo, hi], [lo, hi], linestyle="--", color="black")
    plt.xlabel("Actual easy_run_score")
    plt.ylabel("Predicted easy_run_score")
    plt.title("Predicted vs Actual")
    plt.tight_layout()
    plt.show()

    # Residuals
    residuals = y_test - y_pred
    plt.figure(figsize=(8, 4))
    sns.histplot(residuals, kde=True, color="indianred")
    plt.title("Residual Distribution")
    plt.xlabel("Residual (actual - predicted)")
    plt.tight_layout()
    plt.show()

    # Feature importance export
    model = model_result["model"]
    fi = pd.DataFrame({
        "feature": X.columns,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)
    display(fi.head(20))

    fi_path = OUTPUT_DIR / "easy_run_baseline_feature_importance.csv"
    pred_path = OUTPUT_DIR / "easy_run_baseline_predictions.csv"

    fi.to_csv(fi_path, index=False)
    pd.DataFrame({
        "actual": y_test.values,
        "predicted": y_pred,
        "residual": residuals,
    }).to_csv(pred_path, index=False)

    print("Saved:")
    print("-", fi_path)
    print("-", pred_path)
else:
    print("No model artifact export because no model was trained.")

## Practical Conclusions Template

Use this after each weekly update.

- This week, EF trend looked: ...
- Decoupling trend looked: ...
- Best run execution pattern was: ...
- Main risk signal was: ...
- Next-run focus (single priority): ...